In [ ]:
import torch
import torch.nn as nn
import librosa
import gradio as gr
import numpy as np
import soundfile as sf
import os
import base64
import random
from transformers import AutoProcessor, AutoModelForCTC, Wav2Vec2Processor, Wav2Vec2ForCTC
from Levenshtein import ratio

from feature_extraction import extract_accuracy_features, extract_fluency_features, extract_prosody_features
from PronunciationModel import PronunciationModel  # PronunciationModel 추가
from asr_visualization_for_realtime_gradio import visualize_asr_analysis  # ASR 정렬 시각화 모듈 추가
from phoneme_feedback import analyze_phoneme_errors  # 피드백 모듈 불러오기


# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 불러오기
processor = AutoProcessor.from_pretrained("slplab/wav2vec2-large-robust_revised_ETRI_Korean_english-pronunciation")
model_phoneme = AutoModelForCTC.from_pretrained("slplab/wav2vec2-large-robust_revised_ETRI_Korean_english-pronunciation").to(device)

asr_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
asr_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h-lv60-self").to(device)

# 여러 문장 및 정답 음소 설정 (추가 문장을 원하는 만큼 넣을 수 있음)
sentences = [
    {
        "sentence": "At this age, the infant begins to react more to visual stimuli",
        "phoneme": "AE T DH IH S EY JH DH AH IH N F AH N T B IH G IH N Z T UW R IY AE K T M AO R T UW V IH ZH W AH L S T IH M Y AH L AY"
    },
    {
        "sentence": "My Mom tries to be cool by saying that she likes all the same things that I do",
        "phoneme": "M AY M AH M T R AY Z T AH B IY K UW L B AY S EY IH NG DH AE T SH IY L AY K S AO L DH AH S EY M TH IH NG Z DH AE T AY D UW"
    },
    {
        "sentence": "I was really excited when I heard that my favorite band was coming to perform in our city next month",
        "phoneme": "AY W AH Z R IH L IY IH K S AY T IH D W EH N AY HH ER D DH AE T M AY F EY V R AH T B AE N D W AH Z K AH M IH NG T UW P ER F AO R M IH N AW R S IH T IY N EH K S T M AH N TH"
    },
    {
        "sentence": "Could you tell me the way to the station",
        "phoneme": "K UH D Y UW T EH L M IY DH AH W EY T UW DH AH S T EY SH AH N"
    },
    {
        "sentence": "I heard there’s a new Italian restaurant downtown",
        "phoneme": "AY HH ER D DH EH R Z AH N UW IH T AE L Y AH N R EH S T AH R AA N T D AW N T AW N"
    },
    {
        "sentence": "I didn’t expect the weather to be so cold today",
        "phoneme": "AY D IH D AH N T IH K S P EH K T DH AH W EH DH ER T UW B IY S OW K OW L D T AH D EY"
    },
    {
        "sentence": "One of the best things about working remotely is that I can travel and explore new places while still doing my job",
        "phoneme": "W AH N AH V DH AH B EH S T TH IH NG Z AH B AW T W ER K IH NG R IH M OW T L IY IH Z DH AE T AY K AE N T R AE V AH L AE N D IH K S P L AO R N UW P L EY S AH Z W AY L S T IH L D UW IH NG M AY JH AA B"
    }
]

# 랜덤으로 하나의 문장을 선택
selected_item = random.choice(sentences)
default_sentence = selected_item["sentence"]
default_phoneme = selected_item["phoneme"]

In [ ]:

# 예측 함수
def predict_scores_with_asr(audio_path, correct_phoneme, correct_sentence):
    #  **16kHz가 아닌 경우에만 변환**
    y, sr = librosa.load(audio_path, sr=None)  # 원본 SR 유지
    if sr != 16000:
        y = librosa.resample(y, orig_sr=sr, target_sr=16000)  # 16kHz 변환
        sf.write(audio_path, y, 16000)  # 변환된 오디오를 다시 저장


    # ASR 예측
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = asr_processor(audio, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = asr_model(inputs.input_values).logits
    emissions = torch.log_softmax(logits, dim=-1)
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = asr_processor.batch_decode(predicted_ids)[0].capitalize()

    # 음소 예측
    phoneme_inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        phoneme_logits = model_phoneme(phoneme_inputs.input_values).logits
    predicted_phoneme_ids = torch.argmax(phoneme_logits, dim=-1)
    predicted_phonemes = processor.batch_decode(predicted_phoneme_ids.cpu())[0]

    # --------------------------
    # 정답 문장과 전사 결과 간의 유사도 확인
    sentence_similarity = ratio(correct_sentence.lower(), transcription.lower())
    # phoneme_similarity = ratio(correct_phoneme.lower(), predicted_phonemes.lower())
    # 평가 기준 설정
    sentence_threshold = 0.7  # 문장 유사도 기준 (기존 0.7 → 0.6으로 낮춤)
    # phoneme_threshold = 0.5  # 음소 유사도 기준 (문장이 틀려도 발음이 비슷하면 평가 가능)
    # 평가 여부 결정
    if sentence_similarity < sentence_threshold:
        # 문장 유사도와 음소 유사도가 모두 낮으면 평가 진행 X
        return None, transcription
    # --------------------------
    
    # 정해진 문장과 유사도가 충분하다면 나머지 평가 진행

    # 평가 모델
    model = PronunciationModel(
                               len(extract_fluency_features(audio_path)),
                               len(extract_prosody_features(audio_path))).to(device)
    state_dict = torch.load("final_new_scores_4FC_best_pronunciation_model.pth")
    model.load_state_dict({k.replace("module.", ""): v for k, v in state_dict.items()})
    model.eval()

    # 평가 요소 관련 특성 추출
    accuracy_features = torch.tensor(extract_accuracy_features(audio_path, correct_phoneme, processor, model_phoneme, device), dtype=torch.float32).unsqueeze(0).to(device)
    fluency_features = torch.tensor(extract_fluency_features(audio_path), dtype=torch.float32).unsqueeze(0).to(device)
    prosody_features = torch.tensor(extract_prosody_features(audio_path), dtype=torch.float32).unsqueeze(0).to(device)

    # 평가 모델 예측 
    with torch.no_grad():
        fluency_score, prosody_score = model(fluency_features, prosody_features)
    
    fluency_score = round(fluency_score.item() / 0.5) * 0.5
    prosody_score = round(prosody_score.item() / 0.5) * 0.5

    # Accuracy Score 직접 계산 (정답 음소 vs 예측 음소)
    accuracy_score = accuracy_features.mean().item() * 5  # 0~5점 범위로 변환
    accuracy_score = round(accuracy_score / 0.5) * 0.5  # 0.5 단위로 반올림
    
    # Completeness Score 계산
    phoneme_similarity = ratio(correct_phoneme.lower(), predicted_phonemes.lower())
    combined_similarity = (sentence_similarity + phoneme_similarity) / 2
    completeness_score = round(combined_similarity * 5, 1)
    completeness_score = round(completeness_score / 0.5) * 0.5
    
    # Final Score 계산
    scores = [accuracy_score, fluency_score, prosody_score, completeness_score]
    final_score = round(0.4 * min(scores) + 0.2 * sum(sorted(scores)[1:]), 2)

    # 피드백 생성
    phoneme_feedback = analyze_phoneme_errors(correct_phoneme.lower(), predicted_phonemes.lower(), correct_sentence)
    phoneme_feedback = "\n".join(phoneme_feedback)  # 리스트를 문자열로 변환
    
    # ASR 정렬 및 시각화 실행
    fig, word_segments_audio = visualize_asr_analysis(audio_path, transcription, emissions)
    
    return accuracy_score, fluency_score, prosody_score, completeness_score, final_score, transcription, predicted_phonemes, phoneme_feedback, fig, word_segments_audio



def get_base64_audio(file_path):
    with open(file_path, "rb") as f:
        audio_bytes = f.read()
    base64_audio = base64.b64encode(audio_bytes).decode("utf-8")
    return f"data:audio/wav;base64,{base64_audio}"


# 새로운 문장을 선택하는 함수
def change_sentence():
    global default_sentence, default_phoneme
    selected_item = random.choice(sentences)
    default_sentence = selected_item["sentence"]
    default_phoneme = selected_item["phoneme"]
    return default_sentence

# Gradio 인터페이스
def evaluate_pronunciation(audio):
    if audio is None:
        return "Error: No audio received. Please check your microphone or re-upload the audio.", None, None
    print(f"Received audio path: {audio}")  # 디버깅용 로그 출력

    if not os.path.exists(audio) or os.path.getsize(audio) == 0:
        return "Error: No valid audio file received.", None, None

    # 파일 경로인 경우 바로 처리
    if isinstance(audio, str):  
        temp_audio_path = audio  # 이미 파일 경로이므로 그대로 사용
    else:
        temp_audio_path = "temp_audio.wav"

        if isinstance(audio, tuple):  # (sr, data) 형태로 오는 경우
            sr, audio = audio
        else:
            sr = 16000  # 기본 16kHz 가정

        if len(audio.shape) > 1:  # 다중 채널 오디오인 경우, 첫 번째 채널만 사용
            audio = audio[0, :]
            
        if not np.issubdtype(audio.dtype, np.floating):
            audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max

        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000, res_type="kaiser_fast")

        sf.write(temp_audio_path, audio, 16000)

    # 평가 함수 호출
    result = predict_scores_with_asr(temp_audio_path, default_phoneme, default_sentence)
    
    # result가 튜플이 아니라 (None, transcription)이면, 정해진 문장이 아닌 경우임
    if result[0] is None:
        return "주어진 문장을 발화해주세요", None, None

    
    accuracy_score, fluency_score, prosody_score, completeness_score, final_score, transcription, predicted_phonemes, phoneme_feedback, fig, word_segments_audio = predict_scores_with_asr(
        temp_audio_path, default_phoneme, default_sentence
    )
    
    result_text = f"""
    Pronunciation Evaluation Results
    - Accuracy Score: {accuracy_score}
    - Fluency Score: {fluency_score}
    - Prosody Score: {prosody_score}
    - Completeness Score: {completeness_score}
    - Final Score: {final_score}

    Transcription Results
    - Given Sentence: {default_sentence}
    - Pronounced Sentence: {transcription}
    - Predicted Phonemes: {predicted_phonemes}

    Phoneme Feedback:
    {phoneme_feedback}
    """

    # Create an HTML snippet for word segments
    word_audio_html = "<div style='display: flex; flex-wrap: wrap;'>"
    
    for label, score, file_path in word_segments_audio:
        abs_path = os.path.abspath(file_path)  # 절대 경로 변환
    
        if os.path.exists(abs_path) and os.path.getsize(abs_path) > 0:
            audio_src = get_base64_audio(abs_path)
            word_audio_html += f"""
            <div style="margin: 10px; text-align: center;">
                <strong>{label} ({score:.2f})</strong><br>
                <audio controls>
                    <source src="{audio_src}" type="audio/wav">
                    Your browser does not support the audio element.
                </audio>
            </div>
            """
        else:
            print(f"Skipping {label}: Missing or empty file {abs_path}")
    
    word_audio_html += "</div>"
    
    return result_text, fig, word_audio_html


In [ ]:
with gr.Blocks(css="""
.output-class {height: 800px !important; width: 1200px !important;}
.image-class {height: 600px !important; width: 1100px !important;} 
""") as interface:

    gr.Markdown("### Pronunciation Assessment with ASR Visualization")

    # 문장 표시
    sentence_display = gr.Markdown(f"**Please read the following sentence aloud:**\n\n**{default_sentence}**")

    with gr.Row():
        audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", interactive=True)

    submit_button = gr.Button("Evaluate")

    # "Change Sentence" 버튼 추가
    change_button = gr.Button("Change Sentence")
    
    # 버튼 클릭 시 새로운 문장으로 변경
    change_button.click(fn=change_sentence, inputs=[], outputs=[sentence_display])

    
    with gr.Row():
        output_text = gr.Textbox(label="Evaluation Results", interactive=False)
        
    with gr.Row():
        output_image = gr.Image(label="Alignment Visualization", elem_id="image-class", type="filepath")

    with gr.Row():
        output_word_audio = gr.HTML(label="Word-Level Audio Segments")


    submit_button.click(fn=evaluate_pronunciation, inputs=audio_input, outputs=[output_text, output_image, output_word_audio])

# Gradio 실행
interface.launch(server_name="0.0.0.0", server_port=7860, share=True)
